# ARO for Data Engineers — the medallion tour

Companion notebook to the book. Every cell below is the same code the
book quotes, in the order it is introduced.

The landing CSVs sit next to this notebook, so it runs
wherever your notebook working directory is.

## 1. Values, records, collections

In [ ]:
Create the <numbers> with [3, 1, 4, 1, 5, 9, 2, 6].
Compute the <total: sum> from the <numbers>.

In [ ]:
Create the <orders> with [
    {order_id: 1001, sku: "SKU-RED", quantity: 2, unit_price: 19.99},
    {order_id: 1002, sku: "SKU-BLU", quantity: 1, unit_price: 49.50}
].

A list of records renders as a table. Reach into one with a qualifier:

In [ ]:
Extract the <head: first> from the <orders>.
Extract the <sku> from the <head: sku>.

## 2. Reading the source

`Read` picks its parser from the file extension.

In [ ]:
Read the <raw-orders> from "./landing/orders.csv".
Compute the <row-count: length> from the <raw-orders>.

Numeric columns arrive usable — no cast, no schema:

In [ ]:
Extract the <first-order: first> from the <raw-orders>.
Extract the <qty> from the <first-order: quantity>.
Extract the <price> from the <first-order: unit_price>.
Compute the <line-total> from <qty> * <price>.

## 3. Silver: filter, then join

Only confirmed orders are revenue.

In [ ]:
Filter the <confirmed> from the <raw-orders> where <status> = "confirmed".
Compute the <confirmed-count: length> from the <confirmed>.

The join is a filter inside a loop — for this order's customer id, find the customer:

In [ ]:
Read the <customers> from "./landing/customers.csv".

for each <order> in <confirmed> {
    Extract the <cid> from the <order: customer_id>.
    Filter the <candidates> from the <customers> where <customer_id> = <cid>.
    Extract the <customer: first> from the <candidates>.

    Extract the <q> from the <order: quantity>.
    Extract the <p> from the <order: unit_price>.
    Compute the <total> from <q> * <p>.

    Log <customer: region> ++ "  " ++ <order: sku> ++ "  " ++ <total> to the <console>.
}

## 4. Gold: distinct, then aggregate

`Map … with <field>` projects one column; `unique` keeps first-seen order.

In [ ]:
Map the <all-regions> from the <customers> with region.
Compute the <regions: unique> from the <all-regions>.

## 5. What `Map` will not do

`Map` takes a field name, not an expression — there is no per-element
binding for one to range over. This cell is expected to fail with
`Undefined variable: unit_price`.


In [ ]:
Map the <discounted> from the <raw-orders> with <unit_price> * 0.9.

Per-element arithmetic goes in a `for each`, as in section 3.

## 6. The whole pipeline

The three stages live in `Examples/MedallionPipeline/main.aro`. Run the
finished thing from a shell:

```bash
aro run ./Examples/MedallionPipeline
```

```
== bronze ==
  bronze/orders.jsonl — 6 rows
  bronze/customers.jsonl — 4 rows
== silver ==
  silver/order_facts.jsonl — 4 rows
== gold ==
  gold/revenue_by_region.csv — 2 rows
[OK] pipeline
```